# UPI Fraud Detection — Exploratory Data Analysis
**Author:** Umesh R Kale | EDXcellence Internship

This notebook explores the synthetic UPI transaction dataset to understand fraud patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Libraries loaded!')

In [ ]:
df = pd.read_csv('../data/upi_transactions.csv', parse_dates=['timestamp'])
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
print('=== Dataset Info ===')
print(df.dtypes)
print(f'\nMissing values:\n{df.isnull().sum()}')
print(f'\nFraud distribution:\n{df.is_fraud.value_counts()}')
print(f'\nFraud ratio: {df.is_fraud.mean()*100:.2f}%')

## 1. Fraud vs Normal — Amount Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Amount distribution
sns.histplot(data=df, x='amount', hue='is_fraud', bins=50,
             ax=axes[0], log_scale=True, palette={0:'steelblue',1:'crimson'})
axes[0].set_title('Amount Distribution: Fraud vs Normal (log scale)')
axes[0].set_xlabel('Transaction Amount (₹)')

# Boxplot
sns.boxplot(data=df, x='is_fraud', y='amount', ax=axes[1],
            palette={0:'steelblue',1:'crimson'})
axes[1].set_title('Amount Spread: Normal (0) vs Fraud (1)')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print('Mean amount — Normal:', df[df.is_fraud==0].amount.mean().round(2))
print('Mean amount — Fraud: ', df[df.is_fraud==1].amount.mean().round(2))

## 2. Fraud by Hour of Day

In [ ]:
df['hour'] = df['timestamp'].dt.hour
hourly = df.groupby('hour')['is_fraud'].agg(['sum','count'])
hourly['fraud_rate'] = (hourly['sum'] / hourly['count'] * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(hourly.index, hourly['sum'], color=[
    'crimson' if (h < 6 or h > 22) else 'steelblue' for h in hourly.index])
axes[0].set_title('Fraud Count by Hour')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Fraud Count')
axes[0].axvspan(-0.5, 5.5, alpha=0.1, color='red', label='High risk window')
axes[0].axvspan(22.5, 23.5, alpha=0.1, color='red')
axes[0].legend()

axes[1].plot(hourly.index, hourly['fraud_rate'], marker='o', color='crimson')
axes[1].set_title('Fraud Rate (%) by Hour')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Fraud Rate (%)')
axes[1].fill_between(hourly.index, hourly['fraud_rate'], alpha=0.2, color='crimson')

plt.tight_layout()
plt.show()

## 3. Fraud by Merchant Type

In [ ]:
merch = df.groupby('merchant_type')['is_fraud'].agg(['sum','count'])
merch['fraud_rate'] = (merch['sum'] / merch['count'] * 100).round(2)
merch = merch.sort_values('fraud_rate', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ['crimson' if r > 20 else 'orange' if r > 10 else 'steelblue'
          for r in merch['fraud_rate']]
axes[0].barh(merch.index, merch['fraud_rate'], color=colors)
axes[0].set_title('Fraud Rate (%) by Merchant Type')
axes[0].set_xlabel('Fraud Rate (%)')

axes[1].barh(merch.index, merch['sum'], color=colors)
axes[1].set_title('Total Fraud Count by Merchant Type')
axes[1].set_xlabel('Fraud Count')

plt.tight_layout()
plt.show()
print(merch[['sum','count','fraud_rate']])

## 4. Fraud Type Breakdown

In [ ]:
fraud_types = df[df.is_fraud==1]['fraud_type'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors = ['#e63946','#f4a261','#2a9d8f','#457b9d']

axes[0].pie(fraud_types.values, labels=fraud_types.index,
            autopct='%1.1f%%', colors=colors, startangle=90)
axes[0].set_title('Fraud Type Distribution')

axes[1].bar(fraud_types.index, fraud_types.values, color=colors)
axes[1].set_title('Fraud Count by Type')
axes[1].set_xlabel('Fraud Type')
axes[1].set_ylabel('Count')
plt.xticks(rotation=15)

plt.tight_layout()
plt.show()
print(fraud_types)

## 5. Correlation Heatmap

In [ ]:
numeric_cols = ['amount','is_night','is_new_merchant','device_change','is_fraud']
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()
print('\nCorrelation with is_fraud:')
print(corr['is_fraud'].sort_values(ascending=False))

## 6. Key Findings

- **Fraud peaks between 1–4 AM** — 3× higher fraud rate than daytime
- **Unknown merchant type** has the highest fraud rate (nearly 100%)
- **Fraudulent amounts** are on average 7–12× higher than normal transactions
- **4 distinct fraud patterns** each have roughly equal representation
- **New merchant + large amount** is the strongest fraud combo signal